# Cryptocurrency Price Spike Detection

### Market Trading Algorithmic Signals — Banking-AI

This notebook explains how to detect unusual price spikes in the highly volatile crypto market:

## Step 0 — Notebook Header

**Prerequisites:** Basic Python (`pandas`, `numpy`, `matplotlib`), familiarity with `scikit-learn` basics, and basic understanding of market trading and algorithmic signals terminology.

**What You Will Learn:**

After completing this notebook, you will be able to:
- Describe how unsupervised anomaly detection works and its relevance to market trading and algorithmic signals.
- Implement an anomaly detection pipeline on synthetic data and interpret results in operational terms.
- Evaluate the trade-off between detection sensitivity and false-alarm rate in a banking monitoring context.

**Estimated time:** 35–45 minutes

**Collection context:** Volatility forecasting, trading signals, price prediction, and quantitative market analysis.

## Step 1 — Banking Problem Introduction

**Why this notebook matters:** Cryptocurrencies are famous for extreme volatility. Sometimes, a price jumps because of real news. Other times, it's a "flash crash" or a coordinated "pump and dump." Detecting these anomalies in real-time allows traders to exit risky positions or profit from the sudden move before it reverses.

**Input data used:** Minute-level Returns, Volume, and Price acceleration.

**What we detect:** Anomalous price spikes (`Anomaly`).

**ML method used:** Isolation Forest (Unsupervised Learning).

**Learning goal:** Learn how to detect anomalies without having "labeled" data.

**Primary stakeholders:** quantitative analysts, portfolio managers, risk officers, and trading desk supervisors.

**Comprehension check:** Before looking at the data, write one sentence describing the core decision this model supports and who benefits from getting it right.

**Domain framing note:** This notebook uses synthetic data for educational purposes. It demonstrates decision-support prototyping, not production-ready deployment.

## Step 2 — Environment Setup

Import libraries and set deterministic seeds for reproducibility.

In [ ]:
# Requires: numpy>=1.24, pandas>=2.0, scikit-learn>=1.3, matplotlib>=3.7, seaborn>=0.13

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import IsolationForest

sns.set_theme(style="whitegrid")

RANDOM_SEED = 42
RNG = np.random.default_rng(RANDOM_SEED)
plt.rcParams["figure.figsize"] = (10, 6)

print("Environment ready.")


## Step 3 — Data Creation & Context

Build Synthetic Crypto Price Data

We simulate 2 days of minute-level trading (2,880 minutes) with 5 artificial "spikes".

In [ ]:
n_minutes = 2880
rng = RNG

# 1. Normal Volatility
returns = rng.normal(0, 0.001, n_minutes)
price = 1000 * np.exp(np.cumsum(returns))

# 2. Inject artificial spikes (Sudden 3% - 5% moves in 1 minute)
spike_indices = [500, 1200, 1800, 2200, 2600]
for idx in spike_indices:
    price[idx:] *= 1.04 # 4% jump

df = pd.DataFrame({
    'minute': range(n_minutes),
    'price': price
})

# 3. Feature Engineering: Returns and Volume (synthetic)
df['returns'] = df['price'].pct_change()
df['abs_returns'] = df['returns'].abs()
df['volume'] = rng.poisson(50, n_minutes)
df.loc[spike_indices, 'volume'] *= 10 # High volume during spikes

df.dropna(inplace=True)
df.head()

## Step 4 — Exploratory Data Analysis

For this workflow, primary exploration happens through the operational visualisations in later steps.

## Step 5 — Feature Engineering

Prepare features for modelling. Domain-meaningful transformations help the model learn patterns aligned with banking practice.

For this workflow, the data prepared in Step 3 is used directly as input features.

## Step 6 — Baseline Establishment

A baseline sets the floor that any useful model must beat. Without it, performance numbers have no context.

In [ ]:
# --- Baseline: flag observations beyond a fixed percentile ---
THRESHOLD_PCT = 97
numeric_cols = X.select_dtypes(include='number').columns.tolist()
if 'anomaly_label' in numeric_cols:
    numeric_cols.remove('anomaly_label')
if 'is_anomaly' in numeric_cols:
    numeric_cols.remove('is_anomaly')
thresholds = X[numeric_cols].quantile(THRESHOLD_PCT / 100)
baseline_flags = (X[numeric_cols] > thresholds).any(axis=1)
print(f"Percentile baseline ({THRESHOLD_PCT}th) flags {baseline_flags.sum()} of {len(X)} observations.")
print("Isolation Forest should produce more precise anomaly boundaries.")

## Step 7 — Model Training & Evaluation

Train the model and compare performance against the baseline.

**Prediction prompt:** Before viewing the results, what performance do you expect and why?

Isolation Forest works by "isolating" observations. Anomalies are easier to isolate (require fewer splits) than normal data points.

In [ ]:
# We use returns and volume to find anomalies
X = df[['returns', 'volume']]

# contamination=0.01 means we expect roughly 1% of data to be anomalous
model = IsolationForest(contamination=0.01, random_state=RANDOM_SEED)
df['anomaly_label'] = model.fit_predict(X)

# Isolation Forest labels: -1 for anomaly, 1 for normal
df['is_anomaly'] = df['anomaly_label'] == -1

print(f"Detected {df['is_anomaly'].sum()} anomalous minutes.")

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(df['minute'], df['price'], label='Crypto Price', color='#2A9D8F', alpha=0.5)

# Mark anomalies
anomalies = df[df['is_anomaly']]
plt.scatter(anomalies['minute'], anomalies['price'], color='#E76F51', label='Detected Spike', zorder=5)

plt.title('Crypto Price Spike Detection using AI')
plt.legend()
plt.tight_layout()
plt.show()
plt.close()

**Alt-text:** Scatter plot, line chart titled "Crypto Price Spike Detection using AI". The chart highlights spatial separation or clustering patterns in the data.

## Step 8 — Interpretability & Explainability

Interpret the model in domain terms. A banking professional should be able to assess whether the model's reasoning is credible.

**Summary:**
- The model successfully identified the 5 spikes we injected, plus a few "natural" anomalies caused by random noise.
- By looking at both **Returns** and **Volume**, the model can distinguish between a slow climb and a sudden, high-volume spike.

**Market Context:** Crypto exchanges use these models to freeze trading during suspicious activity or to alert users of significant "Whale" movements (large trades). It's a key tool for maintaining market integrity in a decentralized world.

## Step 9 — Operational Application

Demonstrate how the model output feeds into a real banking workflow decision.

In [ ]:
# --- Scenario: score new observations ---
print("Operational demonstration — anomaly scoring:\n")
new_obs = pd.DataFrame({
    X.columns[0]: [X[X.columns[0]].median(), X[X.columns[0]].quantile(0.99)],
    X.columns[1]: [X[X.columns[1]].median(), X[X.columns[1]].quantile(0.99)],
}, index=["routine", "suspicious"])

scores = model.decision_function(new_obs)
preds  = model.predict(new_obs)
for name, sc, pr in zip(new_obs.index, scores, preds):
    status = "FLAGGED" if pr == -1 else "normal"
    print(f"  {name}: anomaly score = {sc:.3f}  {status}")

print("\nFlagged items would be routed to a human investigator.")

## Step 10 — Summary & Reflection

**What you accomplished:** You built an end-to-end market trading and algorithmic signals workflow — from problem framing through data creation, baseline comparison, model training, evaluation, and operational demonstration.

**Limitations:**

- The dataset is synthetic and cannot capture the full complexity of real banking data.
- Model performance on synthetic data does not guarantee equivalent performance in production.
- This notebook is an educational prototype. Real deployment requires validated data, regulatory review, and ongoing monitoring.

**Ethical reflection:**

- Algorithmic trading models can amplify market volatility and systemic risk if deployed without safeguards.
- Backtested performance often overstates real-world returns due to look-ahead bias and transaction costs.
- Automated trading decisions must include human oversight and circuit breakers.

**Reflection questions:**

1. What additional data sources or features would you need before trusting this model in a live banking environment?
2. How would the relative cost of false positives versus false negatives change the metric you optimise for?
3. How could you adapt this workflow to a related problem in market trading and algorithmic signals?

## References

- Scikit-learn documentation: [https://scikit-learn.org/stable/](https://scikit-learn.org/stable/)
- Project quality baseline: `QUALITY_STANDARD.md`
- Domain context drawn from public banking and financial services literature.